# Structural Damage Classification - Submission

Submission memakai **LightGBM Prefix8 Original** (200 trees) tanpa probability adjustment.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "scripts").is_dir():
    raise FileNotFoundError("Jalankan notebook dari folder project atau folder notebooks.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data" / "raw"
for filename in ("train.csv", "test.csv", "sample_submission.csv"):
    if not (DATA_DIR / filename).is_file():
        raise FileNotFoundError(f"File belum ditemukan: {DATA_DIR / filename}")

In [ ]:
import time
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from scripts.train import train_final_lightgbm
from scripts.predict import make_submission

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

In [ ]:
target = "damage_grade"
id_col = "building_id"
random_state = 42
label_order = ["Grade 1", "Grade 2", "Grade 3", "Grade 4", "Grade 5"]
grade_to_int = {label: idx for idx, label in enumerate(label_order)}
numeric_cols = ["count_floors_pre_eq", "age_building", "plinth_area_sq_ft", "height_ft_pre_eq"]
categorical_cols = ["district_id", "vdcmun_id", "ward_id", "land_surface_condition",
    "foundation_type", "roof_type", "ground_floor_type", "other_floor_type",
    "position", "plan_configuration", "superstructure"]
feature_cols = [col for col in train.columns if col not in [id_col, target]]
final_n_estimators = 200
assert set(numeric_cols + categorical_cols) == set(feature_cols)
assert set(train[target].unique()) == set(label_order)
assert set(test.columns) == set(feature_cols + [id_col])

In [ ]:
model_path = ROOT / "models" / "lightgbm_prefix8_final.pkl"
if model_path.exists():
    final_lgb_prefix8 = joblib.load(model_path)
    print("Model loaded:", model_path)
else:
    print("Model belum tersedia. Training LightGBM final...")
    start = time.perf_counter()
    final_lgb_prefix8 = train_final_lightgbm(
        train, target, feature_cols, categorical_cols, grade_to_int,
        random_state, final_n_estimators)
    print("Final training time:", round((time.perf_counter() - start) / 60, 2), "minutes")
    joblib.dump(final_lgb_prefix8, model_path)
    print("Saved:", model_path)

assert final_lgb_prefix8.named_steps["model"].n_estimators == final_n_estimators
assert final_lgb_prefix8.named_steps["features"].feature_cols == feature_cols
assert final_lgb_prefix8.named_steps["features"].categorical_cols == categorical_cols

## 19. Buat dan validasi submission

Mengikuti `sample_submission.csv`, `damage_grade` diisi dengan angka 1 sampai 5 dan file disimpan menggunakan `index=False`. ID disusun kembali mengikuti urutan sample submission.

In [ ]:
submission = make_submission(final_lgb_prefix8, test, sample_submission, target, id_col)
submission_path = ROOT / "submissions" / "submission_prefix8_original.csv"
submission.to_csv(submission_path, index=False)
print("Submission saved:", submission_path)
print("Submission shape:", submission.shape)
print("Missing values:", int(submission.isna().sum().sum()))
display(submission.head())
display(submission[target].value_counts().sort_index().rename("count").to_frame())
print("Submission checks passed.")